In [1]:
#Deep Graph Anomaly Detection with Contrastive Learning

In [3]:
import torch
torch.mps.empty_cache()

In [ ]:
import numpy as np
import h5py
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
file = h5py.File("quark-gluon_data-set_n139306.hdf5", "r")
print(list(file.keys()))

In [ ]:
images = file["X_jets"][:20000]   # load only 20k samples
labels = file["y"][:20000]

print(images.shape)

In [ ]:
images = np.transpose(images, (0,3,1,2))

print(images.shape)

In [ ]:
images = images / np.max(images)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42
)

In [ ]:
def image_to_pointcloud(image):

    nodes = []

    for c in range(3):
        for i in range(125):
            for j in range(125):
                val = image[c,i,j]
                if val > 0:
                    x = i / 125
                    y = j / 125
                    nodes.append([x, y, val, c])

    nodes= np.array(nodes)
    if len(nodes) > max_nodes:
        idx = np.argsort(nodes[:,2])[-max_nodes:]
        nodes = nodes[idx]

    return nodes

In [ ]:
from sklearn.neighbors import NearestNeighbors

def build_edges(nodes, k=10):

    coords = nodes[:, :2]

    nbrs = NearestNeighbors(n_neighbors=k).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    edges = []

    for i in range(len(nodes)):
        for j in indices[i]:
            edges.append([i, j])

    edge_index = torch.tensor(edges).t().contiguous()

    return edge_index

In [ ]:
from torch_geometric.data import Data

def create_graph(image, label):
    nodes = image_to_pointcloud(image)
    x = torch.tensor(nodes, dtype=torch.float)
    edge_index = build_edges(nodes)
    y = torch.tensor([int(label)], dtype=torch.long) 

    return Data(x=x, edge_index=edge_index, y=y)

In [ ]:
graphs = []

for i in range(5000):
    graph = create_graph(images[i], labels[i])
    graphs.append(graph)

In [ ]:
train_graphs, test_graphs = train_test_split(
    graphs, test_size=0.2, random_state=42
)

In [ ]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_graphs, batch_size=16, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=16)

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class GraphEncoder(torch.nn.Module):

    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(4, 64)
        self.conv2 = GCNConv(64, 64)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return x

In [ ]:
class ContrastiveModel(torch.nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = GraphEncoder()
        self.projector = torch.nn.Sequential(
            torch.nn.Linear(64, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 16)
        )

    def forward(self, data):
        h = self.encoder(data)
        z = self.projector(h)
        return z

In [ ]:
def augment_graph(data):
    x = data.x.clone()
    noise = torch.randn_like(x) * 0.01
    data_aug = data.clone()
    data_aug.x = x + noise
    return data_aug

In [ ]:
def contrastive_loss(z1, z2):
    return 1 - torch.cosine_similarity(z1, z2).mean()

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = ContrastiveModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(20):

    total_loss = 0

    for data in train_loader:

        data = data.to(device)

        aug1 = augment_graph(data)
        aug2 = augment_graph(data)

        z1 = model(aug1)
        z2 = model(aug2)

        loss = contrastive_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch, "Loss:", total_loss)

In [ ]:
def get_embeddings(loader):

    model.eval()

    embeddings = []
    labels_list = []

    for data in loader:

        data = data.to(device)

        h = model.encoder(data)

        embeddings.append(h.cpu())
        labels_list.append(data.y.cpu())

    return torch.cat(embeddings), torch.cat(labels_list)

In [ ]:
train_emb, train_labels = get_embeddings(train_loader)
test_emb, test_labels = get_embeddings(test_loader)

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression()
clf.fit(train_emb.detach().numpy(), train_labels.detach().numpy())
pred = clf.predict(test_emb.detach().numpy())

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(test_labels, pred)
print("accuracy:", accuracy)

In [ ]:
from sklearn.metrics import roc_auc_score
probs = clf.predict_proba(test_emb.detach().cpu().numpy())[:,1]
auc = roc_auc_score(test_labels, probs)
print("ROC AUC:", auc)